In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import sys
sys.path.insert(0, '/home/kat/Repos/SALSA/')

In [2]:
from IPython.display import Audio 
from IPython.core.display import display, HTML
def meow():
    display(Audio( url='https://www.wavsource.com/snds_2020-10-01_3728627494378403/animals/cat_y.wav', 
                  embed=True,autoplay=True))

In [3]:
# meow()

### 1) Compute properties from a sample of ChEMBL.

In [4]:
import pandas as pd
from rdkit.Chem import Descriptors
import torch
from rdkit import Chem
from rdkit.ML.Descriptors.MoleculeDescriptors import MolecularDescriptorCalculator

# This is ALL properties in Descriptors
desc_names = sorted([x[0] for x in Descriptors._descList])

## These are just the ones listed out in MolCLR that are releant imho.
c = pd.read_csv('selected_props_for_mahalanobis.csv')
prop_names = c['prop_name'].values.tolist()
print(len(prop_names))

def get_props(smiles, prop_list):
    calc = MolecularDescriptorCalculator(prop_list)
    props = calc.CalcDescriptors(Chem.MolFromSmiles(smiles))
    return torch.nan_to_num(torch.tensor(props),0.0).float()

90


In [27]:
import os
import numpy as np
from tqdm.notebook import tqdm
from property_predictors import surface_predictor
seedy = 666

# # # # # #
size = 10000 
# # # # # #

## Load ChEMBL.
ddir = '/home/kat/Repos/SALSA/data/'
df = pd.read_csv(os.path.join(ddir,'chembl_valid_lte_120.csv'))

## Get sample...
df_samp = df.sample(size, random_state=seedy)
df_samp = df_samp.reset_index(drop=True)
# display(df_samp)

## Calculate MolBert properties on sample.
all_props = []
for sm in tqdm(df_samp.smiles.values,total=len(df_samp)):
    props = get_props(sm, prop_list=prop_names) # surface_predictor(sm)
    all_props.append(props)
all_props = np.vstack([np.array(x) for x in all_props])

  0%|          | 0/10000 [00:00<?, ?it/s]

### 2) Get properties for all training set compounds.

In [28]:
from contra_seq_dataset import get_dataset_array, get_anc_map
from joblib import Parallel, delayed

## Load in training set.
home = '/home/kat/Repos/SALSA/'
anc_path = f'{home}data/model_ready/01/train/anchor_smiles.csv'
aug_path = f'{home}data/model_ready/01/train/augmented_smiles.csv'
df = get_dataset_array(anc_path, aug_path)
anc_map = get_anc_map(df)

## Calculate props for the training set.
parallelizer = Parallel(n_jobs=-1, backend= 'multiprocessing' )
tasks = (delayed(get_props)(sm, prop_names) for sm in df.smiles.values)
_train_props = parallelizer(tasks)
_train_props = np.vstack([np.array(x) for x in _train_props])

### 3) Get rid of all-zero and inf-containing properties in both the training set and chembl.

In [29]:
## Get rid of all-zero properties in chembl set. (to ensure det!=0)
to_drop = []
for i in range(all_props.shape[1]):
    s = sum(all_props[:,i])
    if s==0:
        to_drop.append(i)
    if np.isinf(_train_props[:,i]).any():
        to_drop.append(i) 
    if len(set(all_props[:,i]))==1:
        to_drop.append(i)

## Get rid of properties with weird values (inf, nan, etc.)
for i in range(_train_props.shape[1]):
    if np.isinf(_train_props[:,i]).any():
        to_drop.append(i) 
to_keep = [i for i in range(all_props.shape[1]) if i not in to_drop]
print(len(set(to_drop)), len(to_keep))

6 84


### 4) Slice out valid prop array for both chembl and train,  then compute covariance matrix.

In [32]:
## Training set props
_train_props = np.vstack([np.array(x) for x in _train_props])
train_props = _train_props[:,to_keep]
## ChEMBL props
chembl_props = all_props[:,to_keep]

## Calculate covariance matrix.
cov = np.cov(chembl_props.T)

### Check covariance matrix for "bad" values ...

In [33]:
to_drop = []
for i in range(cov.shape[0]):
    if np.isinf(cov[:,i]).any():
        to_drop.append(i) 
    if len(set(all_props[:,i]))==1:
        to_drop.append(i)
to_keep = [i for i in range(cov.shape[0]) if i not in to_drop]
print(f'Drop: {len(set(to_drop))} Keep: {len(to_keep)}')

Drop: 2 Keep: 80


In [34]:
cov_valid = cov[to_keep,:][:,to_keep]
inv_cov = np.linalg.inv(cov_valid) # inverse of cov matrix 
print(inv_cov.shape)

(80, 80)


In [36]:
chembl_props_valid = chembl_props[:,to_keep] 
train_props_valid = train_props[:,to_keep]
print(inv_cov.shape,chembl_props_valid.shape,train_props_valid.shape)

(80, 80) (10000, 80) (120000, 80)


### <font color=blue> Check that covariance matrix is positive (semi-)definite ?????

In [37]:
def is_pos_def(x):
    return np.all(np.linalg.eigvals(x) > 0)
def is_pos_semidef(x):
    return np.all(np.linalg.eigvals(x) >= 0)
print("Positive definite?",is_pos_def(inv_cov))

Positive definite? False


In [38]:
idc_nonpos_eigs = np.where(np.linalg.eigvals(inv_cov) <= 0)[0]
print(idc_nonpos_eigs)

[ 2 48 49]


In [39]:
np.linalg.eigvals(inv_cov)[idc_nonpos_eigs]

array([-5.44597263e+14+0.j        , -1.45848427e-02+0.02919674j,
       -1.45848427e-02-0.02919674j])

In [18]:
np.iscomplex(inv_cov).any()

False

In [40]:
np.array_equal(inv_cov,inv_cov.T)

False

In [23]:
np.array_equal(cov,cov.T)

True

### <font color=blue> Get nearest positive semidefinite matrix ?

In [59]:
# https://stackoverflow.com/questions/10939213/how-can-i-calculate-the-
# ... nearest-positive-semi-definite-matrix

# Nope ... this returns a metric shit tonne of complex numbers ... lol ... 
def nearPSD(A,epsilon=0):
    n = A.shape[0]
    eigval, eigvec = np.linalg.eig(A)
    val = np.matrix(np.maximum(eigval,epsilon))
    vec = np.matrix(eigvec)
    T = 1/(np.multiply(vec,vec) * val.T)
    T = np.matrix(np.sqrt(np.diag(np.array(T).reshape((n)) )))
    B = T * vec * np.diag(np.array(np.sqrt(val)).reshape((n)))
    out = B*B.T
    return(out)

inv_cov_psd = nearPSD(inv_cov_valid)
print(inv_cov_psd.shape)
print("Positive definite?",is_pos_def(inv_cov_psd))

(82, 82)
Positive definite? False


In [62]:
to_drop = []
for i in range(inv_cov_psd.shape[0]):
    if np.isinf(inv_cov_psd[:,i]).any():
        to_drop.append(i) 
to_keep = [i for i in range(inv_cov_psd.shape[0]) if i not in to_drop]
print(f'Drop: {len(set(to_drop))} Keep: {len(to_keep)}')

Drop: 0 Keep: 82


In [64]:
np.iscomplex(inv_cov_psd).any()

True

## <font color=red> Next ToDo: ..... no idea ... i guess just google more ways of finding nearest PSD matrix? but for now, this is getting dropped for the most part. <br> <br> Only thing left to do I guess is to go back to the original 49 properties and see how many "undefines" i got in the distance computation. ... 

In [60]:
print(np.isnan(train_props).any())
print(np.min(train_props), np.max(train_props))

False
-22.509264 790.717


### 5) Get Mahalanobis distances between each anc-ang pair in the training set w.r.t the previously calculated ChEMBL property covariance matrix.

In [61]:
from scipy.spatial import distance
from itertools import combinations as combo
# np.seterr(all='print')

# # # # # # # # # # # # # #
far_thresh = 8
which_cov = inv_cov_psd
# # # # # # # # # # # # # #

paired_dists = np.zeros((len(df), len(df)))
anc_aug_dists = []
far_pairs = []

for anc,augs in tqdm(anc_map.items(), total=len(anc_map)):
        
    augs = list(augs)
    augs.pop(0)
    
    prop_anc = train_props_valid[anc]    
    for aug in augs:
        
    
        prop_aug = train_props_valid[aug]
        d = distance.mahalanobis(prop_anc, prop_aug, VI=which_cov)
        
        paired_dists[anc][aug] = d
        
        
        if d > far_thresh:
            far_pairs.append([anc,aug])
        anc_aug_dists.append(d)
        
dist_arr = np.array(anc_aug_dists)  
print('Far distance count:',len(far_pairs))
print('Nan count:',np.count_nonzero(np.isnan(dist_arr)))

  0%|          | 0/20000 [00:00<?, ?it/s]

ValueError: setting an array element with a sequence.

In [ ]:
print(min(dist_arr))
print(max(dist_arr))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style='ticks',font_scale=1.25,palette='muted')

plt.figure(figsize=(20,5))
sns.histplot(dist_arr, fill=False, element="step", kde=False) #, bw_adjust=.25)
plt.show()

### 6) Look at some percentiles cutoffs.

In [ ]:
# # # # # # # #
p = 90
# # # # # # # #

dist_arr_real = dist_arr[~np.isnan(dist_arr)]
cutoff = np.percentile(dist_arr_real, p)
cutoff

### 7) Investigate "different" anc-aug pairs.

In [ ]:
far_anc_to_augs = {}
for anc,aug in far_pairs:
    if anc in far_anc_to_augs.keys():
        far_anc_to_augs[anc].append(aug)
    else:
        far_anc_to_augs[anc] = [aug]
        
far_ancs = far_anc_to_augs.keys()
df_far_ancs = pd.DataFrame(far_ancs,columns=['idx'])
df_far_ancs['smiles'] = df_far_ancs.apply(lambda x: df.smiles.values[x])
df_far_ancs['len'] = df_far_ancs.smiles.apply(lambda x: len(x))
df_far_ancs = df_far_ancs.sort_values(by='len', ascending=True, ignore_index=True)
df_far_ancs

In [ ]:
from rdkit.Chem import AllChem
from rdkit import Chem, Geometry
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole

def sm_and_mol(sm):
    mol = Chem.MolFromSmiles(sm)
    sm = Chem.MolToSmiles(mol)
    display(sm,mol)
    
smis = df.smiles.values

for i,row in df_far_ancs.iterrows():
    
    anc_idx = row.idx
    anc_smi = smis[anc_idx]
    
    mols = []
    aug_idc = anc_map[anc_idx]
    for aug_idx in aug_idc:
        print(paired_dists[anc_idx][aug_idx])
        aug_smi = smis[aug_idx]
        mols.append(Chem.MolFromSmiles(aug_smi))
    img = Draw.MolsToGridImage(mols, molsPerRow=len(mols), subImgSize=(250, 250))
    display(img)
    
    mols = [Chem.MolFromSmiles(anc_smi)]
    far_aug_idc = far_anc_to_augs[anc_idx]
    for aug_idx in far_aug_idc:
        print(paired_dists[anc_idx][aug_idx])
        aug_smi = smis[aug_idx]
        mols.append(Chem.MolFromSmiles(aug_smi))
    img = Draw.MolsToGridImage(mols, molsPerRow=len(mols), subImgSize=(250, 250))
    display(img)
    
    print("- - - - - - - - - - - - - - - - - -")